In [ ]:
# Title: 05_Xu_BayesAge_calibration.ipynb
# Author: Lajoyce Mboning
# Date:2025
# Related publication: Emma K. Costa, and Jingxun Chen, in prep


# Description - Run the BayesAge clock on Xu et al

# Set Up

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LassoCV
from numpy import arange
from numpy import absolute
from numpy import mean
from numpy import std
import numpy as np
from scipy import stats
from sklearn.metrics import mean_absolute_error as mae
import statsmodels.api as sm
from scipy.stats import spearmanr
lowess = sm.nonparametric.lowess
from scipy.stats import poisson
from scipy.stats import pearsonr
from datetime import datetime
import time
import random
import itertools

In [ ]:
# set working dir
os.chdir("/labs/twc/Emma/atlas_gdrive_backup/revisions/CodeCheck_revisions/Xu_model_outputs/BayesAge/")

In [ ]:
# List of directories you want to create
dirs_to_create = [
    "./reference/",
    "./predictions/",
    "./gene_sets/",
    "./plots/",
    "./predictions_exhaustive/",
    "./reference_exhaustive/"
]


for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)  # exist_ok=True won't raise an error if it already exists

In [ ]:
def transcriptome_reference(training_matrix,
                        reference_name,
                        output_path="./predictions/",
                        age_prediction="list",
                        age_list=[],
                        min_age=-20,
                        max_age=100,
                        age_step=1,
                        tau=0.7,
                        ):

    start_time = time.time()

    training_cv_df = training_matrix

    training_df_frequency = training_cv_df.iloc[:,:-1]

    #print(training_df_frequency)

    training_df_frequency = training_df_frequency.div(training_df_frequency.sum(axis=1), axis=0)

    genes = training_cv_df.iloc[:,:-1].columns.tolist()

    age_steps = np.arange(min_age, max_age + age_step, age_step)

    # Age values from the reference data
    age_training = training_cv_df["age"].values.flatten()

    final_training_matrix = []

    if age_prediction == "list":

        for gene in genes:
            # Frequency level of current gene
            gene_frequency = training_df_frequency[gene].values.flatten()

            # Perform lowess smoothing
            predicted_frequency_levels = lowess(gene_frequency, age_training, xvals=age_list, frac=tau)

            # Calculate Spearman rank correlation
            rho, p = spearmanr(age_training, gene_frequency)

            spearman_rank = rho

            p_value = p

            final_gene_frequency_tuple = (gene, ) + tuple(predicted_frequency_levels) + (spearman_rank, p_value)

            final_training_matrix.append(final_gene_frequency_tuple)

        df_columns = ["Gene"] + [str(num) for num in age_list] + ["spearman_rank", "p_value"]

    elif age_prediction == "steps":

        for gene in genes:
            # Frequency level of current gene
            gene_frequency = training_df_frequency[gene].values.flatten()

            # Perform lowess smoothing
            predicted_frequency_levels = lowess(gene_frequency, age_training, xvals=age_steps, frac=tau)

            # Calculate Spearman rank correlation
            rho, p = spearmanr(age_training, gene_frequency)

            spearman_rank = rho

            p_value = p

            final_gene_frequency_tuple = (gene, ) + tuple(predicted_frequency_levels) + (spearman_rank, p_value)

            final_training_matrix.append(final_gene_frequency_tuple)

        df_columns = ["Gene"] + [str(num) for num in age_steps] + ["spearman_rank", "p_value"]


    df = pd.DataFrame(final_training_matrix, columns=df_columns)
    df.set_index("Gene", inplace=True)

    # Create output directory if it doesn't exist
    os.makedirs(output_path, exist_ok=True)

    # Save the DataFrame to a TSV file
    df.to_csv(os.path.join(output_path, reference_name), sep="\t")
    print(f"\nReference model dataset written to '{os.path.join(output_path, reference_name)}'")

    number_of_samples = training_cv_df.shape[0]

    # Write report file detailing input matrix
    report_file_path = os.path.join(output_path, f'{reference_name}.report.txt')
    with open(report_file_path, 'w') as writer:
        writer.write("BayesAge reference report for %s\n" % reference_name)
        now = datetime.now()
        write_datetime = now.strftime("%m/%d/%Y %H:%M:%S")
        writer.write("Reference file created: %s\n\n" % write_datetime)
        writer.write("Number of input samples = %s\n" % number_of_samples)
        writer.write("Number of input Genes = %s\n" % len(genes))
        if age_prediction == "list":
            writer.write("Predictions made for ages: %s\n" % age_list)
        elif age_prediction == "steps":
            writer.write("Predictions made for age steps from %s to %s with step %s\n" % (min_age, max_age, age_step))

    print(f"Report file generated at '{report_file_path}'")
    print("-----------------------------------------------------\n\n")

    end_time = time.time()
    elapsed_time = end_time - start_time
    print("\nTime to run construct reference: %0.3f seconds" % elapsed_time)
    print("\nThe reference is constructed and saved!")

    return df  # Optionally return the DataFrame

# Load & format atlas data

In [ ]:
# load raw counts
df_unnorm = pd.read_csv("../../AtlasFiles_forLajoyce_240507/Counts_Atlas_allbatches_merged_v3.csv")
df_unnorm.head(3)

In [ ]:
df_unnorm.rename(columns={"Unnamed: 0": "sample"}, inplace=True)
df_unnorm.set_index("sample", inplace=True)
df_unnorm.head(3)

In [ ]:
df_unnorm = df_unnorm.T
df_unnorm.head(3)

In [ ]:
# load metadata
df_metadata = pd.read_csv("../../AtlasFiles_forLajoyce_240507/ExperimentDesign_allbatches_combined_v7.csv")

df_metadata.head(3)

In [ ]:
df_metadata.reset_index()

In [ ]:
df_metadata.rename(columns={"Unnamed: 0": "sample"}, inplace=True)

df_metadata.head(3)

In [ ]:
df_metadata.set_index("sample", inplace=True)
df_metadata.head(2)

In [ ]:
# Merge based on index
combined_df = pd.concat([df_unnorm, df_metadata[['sex', 'tissue', 'age_days']]], axis=1)
combined_df.rename(columns={"age_days": "age"}, inplace=True)
combined_df.head(3)

# Load & format intervention data

In [ ]:
#raw counts from intervention dataset
df_intervention = pd.read_csv("../../Data_from_Others/AlanXu_Aging/Counts_AlanXu_250410.csv")
df_intervention.head(3)

In [ ]:
df_intervention = df_intervention.T
df_intervention.head(2)

In [ ]:
df_intervention_metadata = pd.read_csv("../../Data_from_Others/AlanXu_Aging/ExperimentDesign_AlanXuAging_update_20250629.csv")
df_intervention_metadata.set_index("Sample.Name", inplace=True)
df_intervention_metadata.head(2)

In [ ]:
# Parse the index and extract components
df_intervention["tissue"] = df_intervention_metadata["tissue"].tolist()
df_intervention["AGE"] = df_intervention_metadata["age"].tolist()
df_intervention["sex"] = df_intervention_metadata["sex"].tolist()
df_intervention["training"] = df_intervention_metadata["UsedForTraining"].tolist()
df_intervention.head(2)

In [ ]:
age_mapping = {
    "6w": 42,
    "16w": 112
}

df_intervention["age"] = df_intervention["AGE"].map(age_mapping)
df_intervention.head(2)

In [ ]:
df_intervention['sex'] = df_intervention['sex'].replace('male', 'M')
df_intervention['sex'] = df_intervention['sex'].replace('female', 'F')

# Brain

In [ ]:
select_tissue = 'Brain'

### Sex combo

In [ ]:
df_intervention_combo = df_intervention.loc[
    (df_intervention.iloc[:, -5] == select_tissue)
]
df_intervention_combo.head(3)

In [ ]:
# remove outliers
df_intervention_combo = df_intervention_combo.drop('brain_6')
#df_intervention_metadata = df_intervention_metadata.drop('brain_6')
df_intervention_combo.head(3)

In [ ]:
all_samples = df_intervention_combo.index
all_samples

random.seed(123)

# Separate the samples
young_samples = [s for s in all_samples if s in df_intervention_combo.index and df_intervention_combo.loc[s, 'age'] == 42]
old_samples = [s for s in all_samples if s in df_intervention_combo.index and df_intervention_combo.loc[s, 'age'] ==  112]

# Randomly sample 2 from each
random_young = random.sample(young_samples, 2)
random_old = random.sample(old_samples, 2)

# Combine the result
random_subset = random_young + random_old

#select your training samples
select_train_samples = random_subset

# Select rows in df_intervention_age_counts_female where index is in female_samples
df_intervention_train = df_intervention_combo.loc[
    df_intervention_combo.index.isin(select_train_samples)
]

df_intervention_train

In [ ]:
# get your testing samples
#all not in select_train_samples
test_samples = [s for s in all_samples if s not in select_train_samples]
test_samples

In [ ]:
# Select rows in df_intervention_age_counts_female where index is in female_samples
df_intervention_test = df_intervention_combo.loc[
    df_intervention_combo.index.isin(test_samples)
]

df_intervention_test.head(2)

In [ ]:
# Specify the path for the new directory for the tissue
directory_path = "./brain/"

# Create the directory
if not os.path.exists(directory_path):
  os.mkdir(directory_path)

In [ ]:
#select raw counts
tissue_df_counts = combined_df.loc[
    (combined_df.iloc[:, -2] == select_tissue)
]
tissue_df_counts.head(5)

In [ ]:
df_intervention_train = df_intervention_train[tissue_df_counts.columns]
df_intervention_train.head(2)

In [ ]:
atlas_plus_intervention = pd.concat([tissue_df_counts, df_intervention_train], axis=0)
atlas_plus_intervention.head(3)

In [ ]:
tissue_age_days_value_counts = atlas_plus_intervention["age"].value_counts().index.tolist()

In [ ]:
age_steps = sorted(tissue_age_days_value_counts)

print(age_steps)

In [ ]:
atlas_plus_intervention_age = atlas_plus_intervention.drop(atlas_plus_intervention.columns[[-2,-3]], axis=1)
atlas_plus_intervention_age.head(2)

In [ ]:
#sample generation
for sample in atlas_plus_intervention_age.index.tolist():

    new_df = atlas_plus_intervention_age.drop(sample)

    new_df.to_csv(f"./brain/brain_sample_{sample}.csv", sep="\t")

In [ ]:
new_age_steps = np.arange(min(age_steps), max(age_steps)+1, 1)

In [ ]:
# Specify the path for the new directory for the tissue
directory_path = "./brain/reference"

# Create the directory
if not os.path.exists(directory_path):
  os.mkdir(directory_path)

In [ ]:
transcriptome_reference(training_matrix= atlas_plus_intervention_age,
                            reference_name="killifish_brain_reference_sexcombo",
                            output_path="./brain/reference/",
                            age_prediction="list",
                            age_list=new_age_steps,
                            min_age=1,
                            max_age=24,
                            age_step=1,
                            tau=0.7)

In [ ]:
df_intervention_test_counts = df_intervention_test.drop(df_intervention_test.columns[[-2, -3,-4, -5]], axis=1)
df_intervention_test_counts.head(3)

In [ ]:
gene_number = 15
df_intervention_test_pred = pd.DataFrame(index=df_intervention_test.index, columns=["prediction"])

for sample in df_intervention_test.index.tolist():

    reference_df = pd.read_csv(f"./brain/reference/killifish_brain_reference_sexcombo", sep="\t", index_col=0)

    # Get top genes based on spearman correlation
    top_abs_spearman_corr = reference_df["spearman_rank"].abs().nlargest(gene_number)

    #print(top_abs_spearman_corr)

    top_genes = top_abs_spearman_corr.index.tolist()

    # Save the DataFrame to a CSV file
    # df_top_genes = pd.DataFrame(top_genes, columns=["genes"])
    # gene_num = "".join([str(gene_number), "genes"])
    # out_file_prefix = 'Antebi_male_pred'
    # out_file_name = "_".join([out_file_prefix, gene_num])
    # output_path = './fat/gene_sets'
    # df_top_genes.to_csv(os.path.join(output_path, out_file_name), sep="\t")
    # print(f"List has been written to {out_file_name}")

    total_reads_sample = df_intervention_test_counts.loc[sample].sum()

    list_of_profile_probabilities_per_age = []

    for age in new_age_steps:

        probability_list_one_age = []

        genes_not_defined = []

        for gene in top_genes:

            gene_frequency = reference_df.loc[gene, f"{age}"]

            if gene_frequency < 0:
                gene_frequency = 10**-10

            sample_expected_counts = gene_frequency * total_reads_sample
            sample_observed_counts = df_intervention_test_counts.loc[sample, gene]

            poisson_probability = poisson.pmf(k=sample_observed_counts, mu=sample_expected_counts)

            # Replace NaN poisson probability with a small positive value and log it
            if np.isnan(poisson_probability):
                poisson_probability = 0.001
                genes_not_defined.append(gene)

            # Replace zero poisson probability with 0.001
            if poisson_probability == 0:
                poisson_probability = 0.001

            logP = np.log(poisson_probability)

            probability_list_one_age.append(logP)

        list_of_profile_probabilities_per_age.append(np.sum(probability_list_one_age))

    # Transform into dataframe with age steps
    age_probability_df = pd.DataFrame({"Pr": list_of_profile_probabilities_per_age}, index=new_age_steps)

    # Compute highest likelihood age among age steps
    max_probability_age = round(float(age_probability_df.idxmax().iloc[0]), 2)

    # Compute highest maximum probability
    max_probability = float(age_probability_df["Pr"].max())

    df_intervention_test_pred.loc[sample, "prediction"] = max_probability_age



print(df_intervention_test_pred)

In [ ]:
df_intervention_test_pred["age"] = df_intervention_test["age"].tolist()
df_intervention_test_pred["sex"] = df_intervention_test["sex"].tolist()


df_intervention_test_pred["age_sex"] = (
    df_intervention_test_pred["age"].astype(str) + "_" +
    df_intervention_test_pred["sex"].astype(str)
)

df_intervention_test_pred

In [ ]:
ordered_x = [
    '42_M', '42_F',
    '112_M', '112_F'
]


# Create the box plot
plt.figure(figsize=(10, 6))
box = sns.boxplot(x='age_sex', y='prediction', hue='age', data=df_intervention_test_pred, palette='deep', showfliers = False,
    order = ordered_x)

# Add data points

custom_palette = ['#273276', '#B14325']  # Use desired hex color codes

# Add data points with custom colors
strip = sns.stripplot(
    x='age_sex',
    y='prediction',
    hue='age',
    data=df_intervention_test_pred,
    palette=custom_palette,  # Custom color palette
    dodge=False,
    alpha=0.9,
    jitter=True,
    size=15,
    order = ordered_x
)

# Remove the legend if not needed
#strip.legend_.remove()

# Customize the plot
plt.title("".join(['Brain sex combo', str(gene_number), ' Genes']))
plt.xlabel('Treatment')
plt.ylabel('Prediction')

# Set y-axis limits
plt.ylim(40, 160)

# Remove the grid
plt.grid(False)


# Show the plot
plt.show()

In [ ]:
df_intervention_test_pred.to_csv("./df_prediction_BayesAge_Xu_brain_sexcombo.csv", sep="\t")

### Several gene set sizes

In [ ]:
# Set up ordered x-axis for plots
ordered_x = [
    '42_M', '42_F',
    '112_M', '112_F'
]

# Custom color palette
custom_palette = ['#273276', '#B14325']

# Directory setup
os.makedirs("./predictions", exist_ok=True)
os.makedirs("./plots", exist_ok=True)

# Dictionary to store prediction DataFrames if needed later
all_predictions = {}

# Loop through gene numbers
for gene_number in range(5, 101, 5):
    print(f"Running prediction for top {gene_number} genes...")

    df_intervention_test_pred = pd.DataFrame(index=df_intervention_test.index, columns=["prediction"])

    for sample in df_intervention_test.index.tolist():
        reference_df = pd.read_csv(f"./brain/reference/killifish_brain_reference_sexcombo", sep="\t", index_col=0)
        top_genes = reference_df["spearman_rank"].abs().nlargest(gene_number).index.tolist()

        total_reads_sample = df_intervention_test_counts.loc[sample].sum()
        list_of_profile_probabilities_per_age = []

        for age in new_age_steps:
            log_probs = []
            for gene in top_genes:
                gene_frequency = reference_df.loc[gene, str(age)]
                gene_frequency = max(gene_frequency, 1e-10)  # Ensure >0

                expected = gene_frequency * total_reads_sample
                observed = df_intervention_test_counts.loc[sample, gene]

                poisson_prob = poisson.pmf(k=observed, mu=expected)
                poisson_prob = max(poisson_prob, 0.001)

                log_probs.append(np.log(poisson_prob))

            list_of_profile_probabilities_per_age.append(np.sum(log_probs))

        age_probability_df = pd.DataFrame({"Pr": list_of_profile_probabilities_per_age}, index=new_age_steps)
        max_probability_age = round(float(age_probability_df.idxmax().iloc[0]), 2)
        df_intervention_test_pred.loc[sample, "prediction"] = max_probability_age

    # Add metadata
    df_intervention_test_pred["age"] = df_intervention_test["age"].tolist()
    df_intervention_test_pred["sex"] = df_intervention_test["sex"].tolist()


    df_intervention_test_pred["age_sex"] = (
        df_intervention_test_pred["age"].astype(str) + "_" +
        df_intervention_test_pred["sex"].astype(str)
    )

    # Save predictions
    all_predictions[gene_number] = df_intervention_test_pred.copy()
    df_intervention_test_pred.to_csv(f"./predictions/age_preds_top_{gene_number}_genes.tsv", sep="\t")

    # --- Plot ---
    plt.figure(figsize=(10, 6))

    box = sns.boxplot(
        x='age_sex',
        y='prediction',
        hue='age',
        data=df_intervention_test_pred,
        palette='deep',
        showfliers=False,
        order=ordered_x
    )

    strip = sns.stripplot(
        x='age_sex',
        y='prediction',
        hue='age',
        data=df_intervention_test_pred,
        palette=custom_palette,
        dodge=False,
        alpha=0.9,
        jitter=True,
        size=15,
        order=ordered_x
    )

    # Remove duplicate legend
    strip.legend_.remove()

    plt.title(f"Brain sex combo {gene_number} Genes")
    plt.xlabel('Treatment')
    plt.ylabel('Prediction')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(40, 160)
    plt.grid(False)

    # Save plot
    plt.savefig(f"./plots/xu_brain_sexcombo_top_{gene_number}_genes.svg", dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved plot and prediction for {gene_number} genes.\n")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Ordered x-axis
ordered_x = [
    '42_M', '42_F',
    '112_M', '112_F'
]

# Custom palette
custom_palette = ['#273276', '#B14325']

# Load all prediction files
all_predictions_list = []

for gene_number in range(5, 101, 5):
    file_path = f"./predictions/age_preds_top_{gene_number}_genes.tsv"
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, sep="\t", index_col=0)
        df["gene_number"] = str(gene_number)  # for cleaner facet titles
        all_predictions_list.append(df)
    else:
        print(f"Warning: {file_path} not found. Skipping.")

# Concatenate into a single DataFrame
all_predictions_df = pd.concat(all_predictions_list, axis=0)

# Plot with seaborn FacetGrid (catplot)
g = sns.catplot(
    data=all_predictions_df,
    x="age_sex",
    y="prediction",
    hue="age",
    col="gene_number",
    col_wrap=4,
    kind="box",
    palette='deep',
    order=ordered_x,
    height=4,
    aspect=1.2,
    showfliers=False
)

# Overlay stripplot for better visualization
for ax, gene_number in zip(g.axes.flat, all_predictions_df["gene_number"].unique()):
    subset = all_predictions_df[all_predictions_df["gene_number"] == gene_number]
    sns.stripplot(
        x='age_sex',
        y='prediction',
        hue='age',
        data=subset,
        palette=custom_palette,
        dodge=False,
        alpha=0.9,
        jitter=True,
        size=5,
        order=ordered_x,
        ax=ax,
        legend=False
    )
    ax.set_title(f"Top {gene_number} Genes")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_ylim(40, 160)

g.set_axis_labels("Sample Group", "Predicted Age")
g.set_titles("Top {col_name} Genes")
g.tight_layout()
plt.savefig("./plots/brain_sex_combo_faceted_predictions_by_gene_number.svg", dpi=300, bbox_inches="tight")
plt.show()


### Rotate training samples

In [ ]:
# List of directories you want to create
dirs_to_create = [
    "./predictions_exhaustive/brain/",
    "./reference_exhaustive/brain/"
]


for d in dirs_to_create:
    os.makedirs(d, exist_ok=True)  # exist_ok=True won't raise an error if it already exists

In [ ]:
#------------------------------------------------------------
# Sample selection setup
#------------------------------------------------------------
#select two AL samples for calibration
all_samples = df_intervention_combo.index

# Separate the samples
young_samples = [s for s in all_samples if s in df_intervention_combo.index and df_intervention_combo.loc[s, 'age'] == 42]
old_samples = [s for s in all_samples if s in df_intervention_combo.index and df_intervention_combo.loc[s, 'age'] ==  112]

# Create all 2x2 combinations
young_combos = list(itertools.combinations(young_samples, 2))
old_combos = list(itertools.combinations(old_samples, 2))


#------------------------------------------------------------
# Loop through all young-old combinations
#------------------------------------------------------------
for y_combo in young_combos:
    for o_combo in old_combos:

        combo_id = f"{'_'.join(map(str, y_combo))}__{'_'.join(map(str, o_combo))}"
        print(f"\nRunning combo: {combo_id}")

        # Select samples for training
        select_train_samples = list(y_combo) + list(o_combo)
        df_intervention_train = df_intervention_combo.loc[select_train_samples]
        df_intervention_train = df_intervention_train[tissue_df_counts.columns]  # match columns

        # Combine with atlas
        atlas_plus_intervention = pd.concat([tissue_df_counts, df_intervention_train], axis=0)
        age_steps = sorted(atlas_plus_intervention["age"].value_counts().index.tolist())
        atlas_plus_intervention = atlas_plus_intervention.drop(atlas_plus_intervention.columns[[-2, -3]], axis=1)

        # Get test set
        test_samples = [s for s in df_intervention_combo.index if s not in select_train_samples]
        df_intervention_test = df_intervention_combo.loc[test_samples]
        df_intervention_test_counts = df_intervention_test.drop(
            df_intervention_test.columns[[-2, -3, -4, -5]], axis=1
        )

        # check cols ordered the same
        df_intervention_test_counts = df_intervention_test_counts[atlas_plus_intervention.columns]
        df_intervention_test_counts.head(2)

        # (per-sample CSV write loop REMOVED)

        new_age_steps = np.arange(min(age_steps), max(age_steps) + 1, 1)

        # Run transcriptome_reference
        transcriptome_reference(
            training_matrix=atlas_plus_intervention,
            reference_name=f"killifish_brain_ref_sexcombo_{combo_id}",
            output_path="./reference_exhaustive/brain/",
            age_prediction="list",
            age_list=new_age_steps,
            min_age=1,
            max_age=24,
            age_step=1,
            tau=0.7
        )

        # Load reference ONCE per combo
        reference_path = f"./reference_exhaustive/brain/killifish_brain_ref_sexcombo_{combo_id}"
        reference_df = pd.read_csv(reference_path, sep="\t", index_col=0)

        # Run predictions for multiple gene numbers
        for gene_number in range(5, 21, 5):
            print(f"  gene_number: {gene_number}")
            df_pred = pd.DataFrame(index=df_intervention_test.index, columns=["prediction"])
            top_genes = reference_df["spearman_rank"].abs().nlargest(gene_number).index.tolist()

            for sample in df_intervention_test.index:
                total_reads = df_intervention_test_counts.loc[sample].sum()
                log_sums = []
                for age in new_age_steps:
                    logP = []
                    for gene in top_genes:
                        freq = max(reference_df.loc[gene, str(age)], 1e-10)
                        mu = freq * total_reads
                        obs = df_intervention_test_counts.loc[sample, gene]
                        p = max(poisson.pmf(obs, mu), 0.001)
                        logP.append(np.log(p))
                    log_sums.append(np.sum(logP))

                pred_age = round(float(new_age_steps[np.argmax(log_sums)]), 2)
                df_pred.loc[sample, "prediction"] = pred_age

            # Add metadata and save
            df_pred["age"] = df_intervention_test["age"]
            df_pred["sex"] = df_intervention_test["sex"]

            # Save prediction results
            df_pred.to_csv(
                f"./predictions_exhaustive/brain/brain_sexcombo_pred_top_{gene_number}_{combo_id}.tsv", sep="\t"
            )